# Analise de Resultados dos Experimentos

Este notebook le os CSVs gerados em `project/results/result_XXXX/` e constroi uma tabela resumo com:
- **Gap media +- DP** - gap relativo ao melhor objetivo conhecido (`best_solutions/best_objectives.csv`)
- **Tempo media +- DP** - tempo de execucao em segundos (coluna `exec_time` dos CSVs)

A tabela e agrupada por **dataset**, **instancia** e **algoritmo**.

**Formula do gap:** `gap (%) = (best_objective - objective) / best_objective * 100`

> Valores de gap negativos indicam que o algoritmo superou o melhor objetivo conhecido.

In [33]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Configuracao de caminhos
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent  # sobe da pasta notebooks/ para a raiz
RESULTS_BASE = PROJECT_ROOT / 'project' / 'results'
BEST_OBJ_CSV = PROJECT_ROOT / 'best_solutions' / 'best_objectives.csv'

print('Raiz do projeto :', PROJECT_ROOT)
print('Pasta de resultados:', RESULTS_BASE)
print('Arquivo de melhores objetivos:', BEST_OBJ_CSV)

Raiz do projeto : /home/vinicius/Documents/CEFET/TCC/pfc2
Pasta de resultados: /home/vinicius/Documents/CEFET/TCC/pfc2/project/results
Arquivo de melhores objetivos: /home/vinicius/Documents/CEFET/TCC/pfc2/best_solutions/best_objectives.csv


In [34]:
# Carrega os melhores objetivos conhecidos
df_best = pd.read_csv(BEST_OBJ_CSV)

# Remove o sufixo .txt do nome da instancia para facilitar o join
df_best['instance'] = df_best['instance'].str.replace('.txt', '', regex=False)

print('Melhores objetivos carregados:', len(df_best), 'instancias')
display(df_best.head(10))

Melhores objetivos carregados: 50 instancias


,dataset,instance,best_objective
0,a,instance_0001,15.000000
1,a,instance_0002,2.000000
2,a,instance_0003,12.000000
3,a,instance_0004,3.500000
4,a,instance_0005,177.875000
5,a,instance_0006,691.000000
6,a,instance_0007,392.250000
7,a,instance_0008,162.941176
8,a,instance_0009,4.416667
9,a,instance_0010,17.112245


In [35]:
# Coleta de todos os CSVs de resultados

def carregar_resultados(results_base):
    """
    Percorre cada subpasta result_XXXX dentro de results_base,
    le todos os CSVs de instancia e retorna um DataFrame unificado.

    Estrutura esperada de cada CSV:
        algo_id, run_id, status, objective, items, aisles, exec_time
    """
    registros = []

    result_dirs = sorted(Path(results_base).glob('result_*'))
    if not result_dirs:
        print('[AVISO] Nenhuma pasta result_XXXX encontrada em:', results_base)
        return pd.DataFrame()

    for result_dir in result_dirs:
        # Le o config.json para saber os datasets deste experimento
        config_path = result_dir / 'config.json'
        dataset_map = {}
        if config_path.exists():
            with open(config_path) as f:
                cfg = json.load(f)
            for ds, instances in cfg.get('datasets', {}).items():
                for inst in instances:
                    inst_stem = inst.replace('.txt', '')
                    dataset_map[inst_stem] = ds

        csv_files = sorted(result_dir.glob('instance_*.csv'))
        for csv_file in csv_files:
            instance_stem = csv_file.stem
            dataset = dataset_map.get(instance_stem, 'desconhecido')
            try:
                df_inst = pd.read_csv(csv_file)
                df_inst['dataset']    = dataset
                df_inst['instance']   = instance_stem
                df_inst['result_dir'] = result_dir.name
                registros.append(df_inst)
            except Exception as e:
                print('[ERRO] Falha ao ler', csv_file, ':', e)

    if not registros:
        print('[AVISO] Nenhum CSV de instancia encontrado.')
        return pd.DataFrame()

    return pd.concat(registros, ignore_index=True)


df_raw = carregar_resultados(RESULTS_BASE)
print('Total de linhas carregadas:', len(df_raw))
if not df_raw.empty:
    display(df_raw.head(10))

Total de linhas carregadas: 5700


,algo_id,run_id,status,objective,items,aisles,exec_time,dataset,instance,result_dir
0,simple_random_exact,2,error,0.000000,0,0,0.000000,a,instance_0001,result_0005
1,simple_random_simple,3,feasible,3.777778,68,18,0.004981,a,instance_0001,result_0005
2,simple_random_multi,2,feasible,3.400000,68,20,0.001687,a,instance_0001,result_0005
3,simple_random_simple,5,feasible,2.720000,68,25,0.003622,a,instance_0001,result_0005
4,simple_random_multi,4,feasible,2.720000,68,25,0.001709,a,instance_0001,result_0005
5,simple_random_multi,5,feasible,2.833333,68,24,0.001725,a,instance_0001,result_0005
6,simple_random_exact,3,error,0.000000,0,0,0.000000,a,instance_0001,result_0005
7,simple_random_exact,1,error,0.000000,0,0,0.000000,a,instance_0001,result_0005
8,simple_random_simple,4,feasible,2.615385,68,26,0.004256,a,instance_0001,result_0005
9,simple_random_exact,4,error,0.000000,0,0,0.000000,a,instance_0001,result_0005


In [36]:
# Filtra apenas execucoes bem-sucedidas (status == 'feasible')
if df_raw.empty:
    df_ok = pd.DataFrame()
else:
    df_ok = df_raw[df_raw['status'] == 'feasible'].copy()
    print('Execucoes com status feasible:', len(df_ok), 'de', len(df_raw))
    df_ok['objective'] = pd.to_numeric(df_ok['objective'], errors='coerce')
    df_ok['exec_time'] = pd.to_numeric(df_ok['exec_time'], errors='coerce')

Execucoes com status feasible: 3800 de 5700


In [37]:
# Calcula o gap relativo ao melhor objetivo conhecido (em decimal)
# gap = (best_objective - objective) / best_objective
# Valores negativos indicam que o algoritmo superou o melhor conhecido

if not df_ok.empty:
    df_ok = df_ok.merge(
        df_best[['dataset', 'instance', 'best_objective']],
        on=['dataset', 'instance'],
        how='left'
    )

    n_sem_best = df_ok['best_objective'].isna().sum()
    if n_sem_best > 0:
        print(f'[AVISO] {n_sem_best} linhas sem melhor objetivo conhecido (serao ignoradas no calculo do gap).')

    df_ok['gap'] = (
        (df_ok['best_objective'] - df_ok['objective']) / df_ok['best_objective']
    )

    print('Gap calculado com sucesso.')
    print(df_ok[['dataset', 'instance', 'algo_id', 'objective', 'best_objective', 'gap']].head(10).to_string(index=False))

Gap calculado com sucesso.
dataset      instance              algo_id  objective  best_objective      gap
      a instance_0001 simple_random_simple   3.777778            15.0 0.748148
      a instance_0001  simple_random_multi   3.400000            15.0 0.773333
      a instance_0001 simple_random_simple   2.720000            15.0 0.818667
      a instance_0001  simple_random_multi   2.720000            15.0 0.818667
      a instance_0001  simple_random_multi   2.833333            15.0 0.811111
      a instance_0001 simple_random_simple   2.615385            15.0 0.825641
      a instance_0001 simple_random_simple   3.238095            15.0 0.784127
      a instance_0001 simple_random_simple   3.190476            15.0 0.787302
      a instance_0001  simple_random_multi   3.578947            15.0 0.761404
      a instance_0001  simple_random_multi   3.350000            15.0 0.776667


In [38]:
# Tabela resumo: Gap media +- DP  |  Tempo media +- DP
# Agrupado por: dataset, instance, algo_id

def formata(media, dp, casas=4):
    fmt = '{:.' + str(casas) + 'f}'
    return fmt.format(media) + ' +- ' + fmt.format(dp)


if df_ok.empty or 'gap' not in df_ok.columns:
    print('[INFO] Nenhum dado disponivel para gerar a tabela.')
    tabela_exibir = pd.DataFrame()
else:
    df_valido = df_ok.dropna(subset=['gap', 'exec_time'])
    grupos = df_valido.groupby(['dataset', 'instance', 'algo_id'])

    tabela = grupos.agg(
        gap_media   = ('gap',       'mean'),
        gap_dp      = ('gap',       'std'),
        tempo_media = ('exec_time', 'mean'),
        tempo_dp    = ('exec_time', 'std'),
        n_runs      = ('run_id',    'count'),
    ).reset_index()

    tabela['gap_dp']   = tabela['gap_dp'].fillna(0.0)
    tabela['tempo_dp'] = tabela['tempo_dp'].fillna(0.0)

    tabela['Gap media + DP']  = tabela.apply(lambda r: formata(r['gap_media'],   r['gap_dp']),   axis=1)
    tabela['Tempo media + DP']    = tabela.apply(lambda r: formata(r['tempo_media'], r['tempo_dp']), axis=1)

    tabela_exibir = tabela[['dataset', 'instance', 'algo_id', 'n_runs', 'Gap media + DP', 'Tempo media + DP']].copy()
    tabela_exibir = tabela_exibir.rename(columns={
        'dataset':  'Dataset',
        'instance': 'Instancia',
        'algo_id':  'Algoritmo',
        'n_runs':   'Runs',
    })
    tabela_exibir = tabela_exibir.sort_values(['Dataset', 'Instancia', 'Algoritmo'])

    display(tabela_exibir.reset_index(drop=True))

,Dataset,Instancia,Algoritmo,Runs,Gap media + DP,Tempo media + DP
0,a,instance_0001,simple_asc_multi,5,0.8429 +- 0.0000,0.0018 +- 0.0005
1,a,instance_0001,simple_asc_simple,5,0.8625 +- 0.0000,0.0050 +- 0.0020
2,a,instance_0001,simple_desc_multi,5,0.7167 +- 0.0000,0.0007 +- 0.0004
3,a,instance_0001,simple_desc_simple,5,0.7333 +- 0.0000,0.0013 +- 0.0005
4,a,instance_0001,simple_diff_multi_bigger,5,0.8029 +- 0.0000,0.0008 +- 0.0003
...,...,...,...,...,...,...
755,a,instance_0020,simple_similar_weighted_multi_smaller,5,0.0000 +- 0.0000,0.0001 +- 0.0000
756,a,instance_0020,simple_similar_weighted_simple_bigger,5,0.2000 +- 0.0000,0.0001 +- 0.0000
757,a,instance_0020,simple_similar_weighted_simple_most_shared,5,0.2000 +- 0.0000,0.0001 +- 0.0000
758,a,instance_0020,simple_similar_weighted_simple_random,5,0.2400 +- 0.0894,0.0001 +- 0.0000


In [39]:
# Tabela agregada por Dataset + Algoritmo
# (media geral sobre todas as instancias)
if not df_ok.empty and 'gap' in df_ok.columns:
    df_valido = df_ok.dropna(subset=['gap', 'exec_time'])
    grupos2 = df_valido.groupby(['dataset', 'algo_id'])

    tabela2 = grupos2.agg(
        gap_media   = ('gap',       'mean'),
        gap_dp      = ('gap',       'std'),
        tempo_media = ('exec_time', 'mean'),
        tempo_dp    = ('exec_time', 'std'),
        n_runs      = ('run_id',    'count'),
    ).reset_index()

    tabela2['gap_dp']   = tabela2['gap_dp'].fillna(0.0)
    tabela2['tempo_dp'] = tabela2['tempo_dp'].fillna(0.0)

    # Ordena pelo gap (crescente: menor gap = melhor)
    tabela2 = tabela2.sort_values('gap_media', ascending=True)

    tabela2['Gap media + DP'] = tabela2.apply(lambda r: formata(r['gap_media'],   r['gap_dp']),   axis=1)
    tabela2['Tempo media + DP']   = tabela2.apply(lambda r: formata(r['tempo_media'], r['tempo_dp']), axis=1)

    tabela2_exibir = tabela2[['dataset', 'algo_id', 'n_runs', 'Gap media + DP', 'Tempo media + DP']].rename(columns={
        'dataset':  'Dataset',
        'algo_id':  'Algoritmo',
        'n_runs':   'Total Runs',
    })

    print('Resumo por Dataset e Algoritmo')
    display(tabela2_exibir.reset_index(drop=True))

Resumo por Dataset e Algoritmo


,Dataset,Algoritmo,Total Runs,Gap media + DP,Tempo media + DP
0,a,simple_desc_multi,100,0.6945 +- 0.2707,0.0905 +- 0.1372
1,a,simple_desc_simple,100,0.7047 +- 0.2678,0.2242 +- 0.3910
2,a,simple_diff_multi_bigger,100,0.7159 +- 0.2872,0.1035 +- 0.1572
3,a,simple_diff_weighted_multi_bigger,100,0.7159 +- 0.2872,0.1013 +- 0.1493
4,a,simple_diff_weighted_multi_most_shared,100,0.7236 +- 0.2815,0.1031 +- 0.1506
5,a,simple_diff_multi_most_shared,100,0.7236 +- 0.2815,0.1065 +- 0.1574
6,a,simple_similar_multi_bigger,100,0.7254 +- 0.2593,0.1040 +- 0.1516
7,a,simple_similar_weighted_multi_bigger,100,0.7254 +- 0.2593,0.1128 +- 0.1725
8,a,simple_diff_multi_random,100,0.7294 +- 0.2649,0.1053 +- 0.1498
9,a,simple_diff_weighted_multi_random,100,0.7294 +- 0.2649,0.1015 +- 0.1471


In [40]:
# Exporta as tabelas para CSV
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

if not df_ok.empty and 'gap' in df_ok.columns and not tabela_exibir.empty:
    path1 = OUTPUT_DIR / 'tabela_por_instancia.csv'
    path2 = OUTPUT_DIR / 'tabela_por_dataset.csv'
    tabela_exibir.to_csv(path1, index=False)
    tabela2_exibir.to_csv(path2, index=False)
    print('Tabelas salvas em:')
    print(' ', path1)
    print(' ', path2)
else:
    print('[INFO] Nada exportado - nenhum dado disponivel.')

Tabelas salvas em:
  /home/vinicius/Documents/CEFET/TCC/pfc2/notebooks/output/tabela_por_instancia.csv
  /home/vinicius/Documents/CEFET/TCC/pfc2/notebooks/output/tabela_por_dataset.csv
